<a href="https://colab.research.google.com/github/AndresAvella09/Proyecto_Multimodal/blob/main/notebooks/13_app_buscador.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 13 · App web del proyecto (Streamlit)

La app vive en el repo, en `app/` (Streamlit multipágina): **buscador** (póster, sinopsis o ambos), **clasificador**
de géneros, **desbalance** película por película (apagar ramas, compuerta del C1, dial del C2), **ayer y hoy**
(resultados del 08, 12 y C1-C3) y **espacio 3D**. Este notebook solo prepara los datos y la lanza.

| Sección | Qué hace | Resultado |
|---|---|---|
| 1 | `scripts/exportar_app.py`: catálogo (MM-IMDB + estrenos 2020-2026), vectores del buscador, logits del clasificador, compuertas, mapa 3D y CSV de resultados | carpeta `app_buscador/` en Drive |
| 2 | Revisa que la búsqueda dé lo mismo que en el notebook 11 | — |
| 3 | **Lanza la app desde Colab** con un link público temporal | link `https://….trycloudflare.com` |
| 4 | (Opcional) `scripts/publicar_hf.py`: datos a un repo de datasets y código a un Space de Hugging Face | link `https://huggingface.co/spaces/…` |

**Requisitos:** notebooks 05, 09, 10 y 11 (buscador). **Opcional:** 07 y 12 (estrenos), 06 y C1 (clasificador y
compuertas: `exp2_hoy_artefactos.pt`, `exp4_compuertas.pt`), C2 (`exp5_compuertas_aum.pt`) y 08, C1-C3 (CSV de
resultados). Si falta algo, esa parte de la app se omite. Corre en CPU.

In [ ]:
import os, sys, subprocess
EN_COLAB = 'google.colab' in sys.modules
if EN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    !pip install -q "streamlit==1.67.0" plotly sentence-transformers umap-learn

REPO = '/content/Proyecto_Multimodal'
if os.path.exists(REPO):
    subprocess.run(['git', '-C', REPO, 'pull', '-q'], check=True)
else:
    subprocess.run(['git', 'clone', '-q', 'https://github.com/AndresAvella09/Proyecto_Multimodal.git', REPO], check=True)
sys.path.insert(0, REPO)

BASE = os.environ.get('PM_BASE', '/content/drive/MyDrive/proyecto_multimodal')
APP_DIR = f'{BASE}/app_buscador'            # lo que necesita la app queda aquí

## 1. Exportar los datos de la app

Mismo contenido que exportaba antes este notebook (catálogo, vectores en `float16`, `config.json` con la corrección por
fuente del 12, figura UMAP) más lo de las páginas nuevas: logits calibrados de imagen, texto y fusión por película,
pesos de las cabezas del 06 y de las compuertas del C1/C2, y el mapa 3D (PCA y UMAP). Tarda unos minutos por el UMAP;
con `--sin-umap` es casi inmediato.

In [ ]:
!cd {REPO} && python scripts/exportar_app.py --base "{BASE}"

## 2. Chequeo: ¿la app busca igual que el notebook 11?

Se busca *"una película de terror"* con la función de la app y se cuenta cuántos de los 12 primeros resultados son de
terror, sin entrenar y con la representación entrenada. Con la entrenada debería haber más.

In [ ]:
os.environ['APP_DATOS'] = APP_DIR
sys.path.insert(0, f'{REPO}/app')
import comun                                 # el mismo módulo que usa la app (funciona fuera de Streamlit)

cat, cfg = comun.cargar_catalogo(), comun.cargar_config()
for version in ['Sin entrenar', 'Entrenada']:
    top, _ = comun.buscar('una película de terror', 'Ambos', version, cfg['alfa'], cfg['anio_min'], cfg['anio_max'], 12)
    r = cat.iloc[top]
    print(f"{version:>12}: {r['generos'].str.contains('Horror').sum()}/12 de terror | {', '.join(r['titulo'].head(4))}")

## 3. Lanzar la app desde Colab

Streamlit corre dentro de Colab y **Cloudflare** crea un link público temporal (`https://….trycloudflare.com`) para
abrirla desde cualquier navegador, incluso desde el celular. El link funciona mientras esta sesión de Colab esté abierta.
La primera búsqueda tarda unos segundos porque la app carga los modelos de lenguaje.

In [ ]:
import re, time
if not os.path.exists('/content/cloudflared'):
    !wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x /content/cloudflared
for patron in ['streamlit run', 'cloudflared tunnel']:       # detener una versión anterior, si la había
    subprocess.run(['pkill', '-f', patron])
time.sleep(1)

app_log = open('/content/streamlit.log', 'w')
subprocess.Popen(['streamlit', 'run', 'app/Inicio.py', '--server.port', '8501'], cwd=REPO,
                 env=dict(os.environ, APP_DATOS=APP_DIR), stdout=app_log, stderr=subprocess.STDOUT)
tunel_log = open('/content/cloudflared.log', 'w')
subprocess.Popen(['/content/cloudflared', 'tunnel', '--url', 'http://localhost:8501', '--no-autoupdate'],
                 stdout=tunel_log, stderr=subprocess.STDOUT)

url = None
for _ in range(60):
    time.sleep(1)
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('/content/cloudflared.log').read())
    if m:
        url = m.group(0); break
if url:
    print('La app está en:', url)
    print('(si aparece un error, espera unos 20 segundos y recarga: la app todavía está arrancando)')
else:
    print('No apareció el link: revisa los registros en la celda siguiente')

In [ ]:
# Si algo falla en la app, aquí se ven los errores
print(open('/content/streamlit.log').read()[-3000:])
print(open('/content/cloudflared.log').read()[-1000:])

## 4. (Opcional) Publicarla en Hugging Face Spaces

Para un **link fijo**: los datos (~400 MB) van a un **repo de datasets** y el código a un **Space** (Docker) que los
descarga al arrancar. Pasos:

1. Cuenta en [huggingface.co](https://huggingface.co) y un token con permiso de escritura (*Settings → Access Tokens*).
2. En Colab, panel de la **llave** 🔑 (Secretos): agrega `HF_TOKEN` con ese token y activa "Acceso al notebook".
3. Cambia `USUARIO_HF`, pon `PUBLICAR = True` y corre la celda.
4. El dataset queda **privado**: en el Space, *Settings → Variables and secrets → New secret*, agrega `HF_TOKEN` con
   un token de **lectura**. La primera construcción tarda ~5-10 minutos (pestaña *Logs* del Space).

In [ ]:
USUARIO_HF = 'tu-usuario'          # <- cambia esto
PUBLICAR = False                   # <- cambia a True para publicar

if PUBLICAR:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    !cd {REPO} && python scripts/publicar_hf.py --usuario {USUARIO_HF} --datos "{APP_DIR}"
else:
    print('No se publicó (PUBLICAR = False).')

## Resumen

- `app_buscador/` en Drive tiene los **datos** de la app; el **código** está en el repo (`app/`, `src/`).
- Se abre desde Colab (sección 3) o con link fijo en Hugging Face (sección 4). En la PC:
  `APP_DATOS=<carpeta app_buscador> streamlit run app/Inicio.py` (ver el README).